In [ ]:
import pandas as pd
from experiment import RollingTestAnalyzer

raw_path = r"rolling_test_raw.csv"
history_df = pd.read_csv(r"data\power_daily_raw.csv")

rolling_df = pd.read_csv(raw_path)
first_target = pd.to_datetime(rolling_df["target_date"]).min()

# 首个 test target_date 之前的真实历史序列
history_actuals = history_df.loc[
    pd.to_datetime(history_df["date"]) < first_target,
    "OT",
].to_numpy(dtype=float)
   
seasonality = 7

a_path = r"artifacts\LSTM_both_feat\20260416_154832\rolling_test_raw.csv"
b_path = r"artifacts\NHITS_both_feat\20260416_154728\rolling_test_raw.csv"

a = RollingTestAnalyzer(
    rolling_raw=a_path,
    history_actuals=history_actuals,
    seasonality=seasonality,
)

b = RollingTestAnalyzer(
    rolling_raw=b_path,
    history_actuals=history_actuals,
    seasonality=seasonality,
)

# 1. overall 对比
overall_a = a.overall_metrics()
overall_b = b.overall_metrics()

# 2. horizon-wise 对比（手动看两张表）
horizon_a = a.loss_matrix("horizon")
horizon_b = b.loss_matrix("horizon")

# 3. window-wise 对比
window_a = a.loss_matrix("window")
window_b = b.loss_matrix("window")

# A 相对 B 的 window 统计 + win_rate
summary_a_vs_b = a.loss_summary("window", baseline=b)
summary_b = b.loss_summary("window")







In [18]:
print(f'a:\n{overall_a}')
print(f'b:\n{overall_b}')



a:
MASE       1.992035e+00
RMSE       3.862510e+08
WAPE(%)    1.786112e+01
Bias       1.813523e+08
MAPE(%)    2.425429e+01
Name: overall, dtype: float64
b:
MASE       1.055323e+00
RMSE       2.150323e+08
WAPE(%)    9.462314e+00
Bias       5.616490e+07
MAPE(%)    1.134245e+01
Name: overall, dtype: float64


In [19]:
print('horizon_a:')
horizon_a


horizon_a:


,MAE,RMSE,Bias
horizon,,,
1,2.349734e+08,3.125554e+08,5.169137e+07
2,3.037465e+08,4.228523e+08,2.465785e+08
3,2.767153e+08,3.797123e+08,1.777743e+08
4,2.622812e+08,3.563938e+08,1.393594e+08
5,2.811176e+08,3.838858e+08,1.795622e+08
6,3.104593e+08,4.296525e+08,2.581888e+08
7,2.950970e+08,4.057913e+08,2.163117e+08


In [20]:
print('horizon_b:')
horizon_b

horizon_b:


,MAE,RMSE,Bias
horizon,,,
1,8.307887e+07,1.210646e+08,1.632962e+07
2,1.225205e+08,1.720377e+08,1.741529e+07
3,1.435790e+08,1.872497e+08,1.553355e+07
4,1.646623e+08,2.226697e+08,5.310608e+07
5,1.664084e+08,2.296791e+08,5.750621e+07
6,1.746111e+08,2.569786e+08,1.076669e+08
7,1.858176e+08,2.756520e+08,1.255966e+08


In [21]:
print(f'window_a:')
window_a

window_a:


,MAE,RMSE,MAPE(%)
origin_date,,,
2024-11-10,2.877445e+08,3.042209e+08,17.642041
2024-11-11,3.253326e+08,3.670137e+08,21.256042
2024-11-12,2.993287e+08,3.416061e+08,19.582997
2024-11-13,2.972574e+08,3.299754e+08,19.213683
2024-11-14,2.896758e+08,3.177265e+08,18.653334
...,...,...,...
2025-01-31,6.935404e+08,7.024420e+08,72.093883
2025-02-01,6.153182e+08,6.317142e+08,60.863353
2025-02-02,5.437241e+08,5.639256e+08,50.884612


In [22]:
print('window_b')
window_b

window_b


,MAE,RMSE,MAPE(%)
origin_date,,,
2024-11-10,1.594496e+08,2.003967e+08,9.749640
2024-11-11,2.266633e+08,2.941670e+08,15.057142
2024-11-12,2.284347e+08,2.975328e+08,15.214716
2024-11-13,1.796845e+08,2.478586e+08,12.074278
2024-11-14,1.220247e+08,1.868670e+08,8.397225
...,...,...,...
2025-01-31,1.690078e+08,1.765706e+08,17.213540
2025-02-01,1.409642e+08,1.519138e+08,13.362226
2025-02-02,1.034324e+08,1.113705e+08,9.318360


In [23]:
summary_a_vs_b

,mean,median,std,worst10%,max,win_rate
MAE,2.806272e+08,1.757121e+08,2.452362e+08,8.305016e+08,8.611423e+08,0.183908
RMSE,3.007256e+08,1.885687e+08,2.423921e+08,8.350935e+08,8.647283e+08,0.195402
MAPE(%),2.425429e+01,9.424949e+00,2.962610e+01,9.555759e+01,1.017773e+02,0.195402


In [24]:
summary_b

,mean,median,std,worst10%,max
MAE,1.486683e+08,1.144478e+08,1.238728e+08,4.735153e+08,5.981099e+08
RMSE,1.706506e+08,1.323318e+08,1.308331e+08,5.070321e+08,6.113516e+08
MAPE(%),1.134245e+01,7.253961e+00,1.211695e+01,4.239774e+01,5.853655e+01
